# Entregável 6 — Extração de Atributos (Feature Extraction)

**Disciplina:** Aquisição de Biossinais  
**Equipe:** José Lessa & Matheus Rocha  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL (PhysioNet)  

---

## Objetivo

Extrair descritores matemáticos (features) das janelas segmentadas de ECG para alimentar 
futuros modelos de classificação (Machine Learning). Segundo a literatura, combinamos 
múltiplas representações para capturar toda a dinâmica cardiovascular.

### Domínios Explorados:
1. **Domínio do Tempo:** Estatísticas de amplitude e complexidade morfológica direta (Ex: RMS, Hjorth).
2. **Domínio da Frequência:** Distribuição de potências espectrais via PSD/FFT.
3. **Domínio Tempo-Frequência:** Análise multi-resolução via Transformada Wavelet Discreta (DWT).
4. **Domínio Não-Linear:** Avaliação do caos ecológico e complexidade do sistema (Ex: Entropia de Shannon, Dimensão Fractal).

## 1. Configuração e Imports

In [ ]:
!pip install pywavelets numba -q

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
import pywt
from pathlib import Path
from scipy import stats, signal as sp_signal
import warnings
warnings.filterwarnings('ignore')

# Configurações de plot
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Diretórios
DATA_DIR = Path('../../data/ptb-xl')
FIG_DIR = Path('../figuras')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Configuração concluída.')

## 2. Preparação dos Dados (Pipelines Anteriores)

Carregamos um paciente, limpamos e extraímos uma de suas janelas de 5 segundos para a 
demonstração de cada método.

In [ ]:
def carregar_ecg(ecg_id, df, data_dir, sampling_rate=500):
    col = 'filename_hr' if sampling_rate == 500 else 'filename_lr'
    filename = df.loc[ecg_id, col]
    return wfdb.rdrecord(str(data_dir / filename))

def limpar_sinal_ecg(sinal, fs):
    """Notch 50Hz + Band-pass 0.5-40Hz"""
    b, a = sp_signal.iirnotch(50.0, 30.0, fs)
    s1 = sp_signal.filtfilt(b, a, sinal)
    nyq = 0.5 * fs
    b, a = sp_signal.butter(4, [0.5/nyq, 40.0/nyq], btype='band')
    return sp_signal.filtfilt(b, a, s1)

df_meta = pd.read_csv(DATA_DIR / 'ptbxl_database.csv', index_col='ecg_id')

# Registro de exemplo
ecg_id = df_meta.index[10] 
record = carregar_ecg(ecg_id, df_meta, DATA_DIR, sampling_rate=500)
fs = record.fs
canal_idx = record.sig_name.index('II') # Padrão ouro ritmico

sinal_completo = limpar_sinal_ecg(record.p_signal[:, canal_idx], fs)

# Segmentação (1 Instância de 5 segundos)
janela_exemplo = sinal_completo[int(2.5*fs) : int(7.5*fs)]
t_janela = np.arange(len(janela_exemplo)) / fs

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_janela, janela_exemplo, color='indigo')
ax.set_title('Janela de Exemplo (Para demonstração das Features)', fontweight='bold')
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('Amplitude (mV)')
plt.show()

## 3. Extratores de Features

In [ ]:
def extract_time_features(signal):
    """
    Extrações Lineares no Domínio do Tempo.
    """
    features = {}
    
    # Amplitudes Globais
    features['t_mean'] = np.mean(signal)
    features['t_var'] = np.var(signal)
    features['t_std'] = np.std(signal)
    features['t_rms'] = np.sqrt(np.mean(signal**2))
    features['t_mav'] = np.mean(np.abs(signal)) # Mean Absolute Value
    features['t_ptp'] = np.max(signal) - np.min(signal) # Peak-to-Peak
    
    # Zero Crossing Rate (Taxa de cruzamento do eixo zero)
    zero_crosses = np.where(np.diff(np.sign(signal)))[0]
    features['t_zcr'] = len(zero_crosses) / len(signal)
    
    # Hjorth Parameters (Atividade, Mobilidade, Complexidade) 
    # Muito usuais para EEG, mas captam morfologia do QRS no ECG perfeitamente.
    dx = np.diff(signal)
    ddx = np.diff(dx)
    
    var_x = np.var(signal)
    var_dx = np.var(dx)
    var_ddx = np.var(ddx)
    
    features['hjorth_activity'] = var_x
    features['hjorth_mobility'] = np.sqrt(var_dx / var_x) if var_x > 0 else 0
    
    if features['hjorth_mobility'] > 0:
        features['hjorth_complexity'] = (np.sqrt(var_ddx / var_dx)) / features['hjorth_mobility']
    else:
        features['hjorth_complexity'] = 0
        
    return features

display(pd.Series(extract_time_features(janela_exemplo), name='Exemplo (Tempo)'))

In [ ]:
def extract_frequency_features(signal, fs):
    """
    Extrações no Domínio da Frequência via PSD (Welch).
    """
    features = {}
    
    f, psd = sp_signal.welch(signal, fs=fs, nperseg=min(512, len(signal)))
    
    # Potência global
    total_power = np.sum(psd)
    features['f_total_power'] = total_power
    
    # Frequência Mediana e Frequência de Pico
    features['f_peak'] = f[np.argmax(psd)]
    
    cum_power = np.cumsum(psd)
    median_idx = np.where(cum_power >= total_power / 2)[0]
    features['f_median'] = f[median_idx[0]] if len(median_idx) > 0 else 0
    
    # Band Power (Energia em faixas específicas para ECG estrutural)
    # VLF (Very Low) < 0.5 Hz - Geralmente Baseline wander (a gente removeu aliás)
    # LF (Low) 0.5 - 5 Hz - Onde fica as ondas P e T e parte ST
    # MF (Medium) 5 - 15 Hz - Miolo de energia do QRS
    # HF (High) 15 - 40 Hz - Frestas e slope do R
    
    features['f_power_LF'] = np.sum(psd[(f >= 0.5) & (f < 5)]) / total_power
    features['f_power_MF'] = np.sum(psd[(f >= 5) & (f < 15)]) / total_power
    features['f_power_HF'] = np.sum(psd[(f >= 15) & (f < 40)]) / total_power
    
    return features

display(pd.Series(extract_frequency_features(janela_exemplo, fs), name='Exemplo (Frequência)'))

In [ ]:
def extract_time_frequency_features(signal):
    """
    Extração no domínio misto usando Transformada Wavelet Discreta (DWT).
    """
    features = {}
    
    # Daubechies 4 (db4) é a wavelet clássica que melhor se assemelha ao QRS clnico
    # Fazemos a decomposição em 4 níveis matemáticos
    coeffs = pywt.wavedec(signal, 'db4', level=4)
    
    # coeffs = [cA4, cD4, cD3, cD2, cD1]
    # Onde cA é aproximação (baixas freqs) e cD são os detalhes (altas freqs)
    nomes = ['cA4', 'cD4', 'cD3', 'cD2', 'cD1']
    
    total_energy = sum([np.sum(c**2) for c in coeffs])
    
    for c, nome in zip(coeffs, nomes):
        energy = np.sum(c**2)
        features[f'wt_energy_{nome}'] = energy / total_energy  # Energia relativa da sub-banda
        features[f'wt_std_{nome}'] = np.std(c)
        features[f'wt_mean_{nome}'] = np.mean(c)

    return features

display(pd.Series(extract_time_frequency_features(janela_exemplo), name='Exemplo (Wavelet)'))

In [ ]:
def extract_nonlinear_features(signal):
    """
    Extrações Não-Lineares de complexidade e previsibilidade informacional.
    """
    features = {}
    
    # 1. Entropia de Shannon (Desorganização do histograma de amplitudes)
    hist, _ = np.histogram(signal, bins=100, density=True)
    hist = hist[hist > 0]
    features['nl_shannon_entropy'] = -np.sum(hist * np.log2(hist))
    
    # 2. Dimensão Fractal de Katz (Medida de complexidade/"dobras" da curva)
    L = np.sum(np.abs(np.diff(signal))) # Comprimento de arco da curva
    d = np.max(np.abs(signal - signal[0])) # Distância end-to-end máxima
    N = len(signal)
    features['nl_katz_fd'] = np.log10(L) / np.log10(d) if d > 0 else 0
    
    # 3. Métricas pseudo-Poincaré dos picos da onda
    # Computar na própria série temporal em lag 1
    x_n = signal[:-1]
    x_n1 = signal[1:]
    
    sd1 = np.std(x_n1 - x_n) / np.sqrt(2)
    sd2 = np.std(x_n1 + x_n) / np.sqrt(2)
    
    features['nl_poincare_sd1'] = sd1
    features['nl_poincare_sd2'] = sd2
    features['nl_poincare_ratio'] = sd1 / sd2 if sd2 > 0 else 0
    
    return features

display(pd.Series(extract_nonlinear_features(janela_exemplo), name='Exemplo (Não-Linear)'))

## 4. Pipeline Completo: Extração Massiva da Base

Consolidamos todos os métodos numa mega função `extract_all` e a aplicamos a uma subamostra populacional (
já fatiando nas Janelas de Segmentação construídas no Entregável 5).

In [ ]:
def extract_all_features(signal, fs):
    features = {}
    features.update(extract_time_features(signal))
    features.update(extract_frequency_features(signal, fs))
    features.update(extract_time_frequency_features(signal))
    features.update(extract_nonlinear_features(signal))
    return features

def criar_janelas(sinal, fs, win_sec=5.0, step_sec=2.5):
    win_len = int(win_sec * fs)
    step_len = int(step_sec * fs)
    n_samples = len(sinal)
    n_janelas = int((n_samples - win_len) / step_len) + 1
    
    janelas = []
    for i in range(n_janelas):
        start = i * step_len
        end = start + win_len
        janelas.append(sinal[start:end])
    return np.array(janelas)

# Executando numa sub-amostra considerável para geração do Dataset Ouro (500 pts)
N_AMOSTRA = 500
np.random.seed(42)
ids_amostra = np.random.choice(df_meta.index, size=N_AMOSTRA, replace=False)
dataset_features = []

for idx, ecg_id in enumerate(ids_amostra):
    if idx % 50 == 0:
        print(f'Extraindo Features: {idx}/{N_AMOSTRA}...')
    
    try:
        record = carregar_ecg(ecg_id, df_meta, DATA_DIR, sampling_rate=500)
        fs = record.fs
        
        for ch_idx, ch_name in enumerate(record.sig_name):
            # Extrairemos apenas da clássica DII (Ritmo) e V1 (Morfologia Ventricular Frontal) pra otimização espacial
            if ch_name not in ['II', 'V1']: 
                continue 
                
            sinal_bruto = record.p_signal[:, ch_idx]
            sinal_limpo = limpar_sinal_ecg(sinal_bruto, fs)
            
            # Janelamento (E5)
            janelas = criar_janelas(sinal_limpo, fs, win_sec=5.0, step_sec=2.5)
            
            for i, idx_janela in enumerate(janelas):
                row = {'ecg_id': ecg_id, 'derivacao': ch_name, 'janela_instancia': i+1}
                # Merge dicionarios de features na row base
                row.update(extract_all_features(idx_janela, fs))
                dataset_features.append(row)
                
    except Exception as e:
        # Tratamento seguro caso falta registro
        pass

df_features = pd.DataFrame(dataset_features)
print(f"\nFINALIZADO! Dataset de ML construído com {len(df_features)} instâncias e {len(df_features.columns)-3} features.")

## 5. Exemplos Gráficos Explicativos dos Métodos Extraídos

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
t_jan = np.arange(len(janela_exemplo))/fs

# Explicativo do RMS (Tempo)
rms_val = np.sqrt(np.mean(janela_exemplo**2))
ax = axes[0, 0]
ax.plot(t_jan, janela_exemplo, color='gray', alpha=0.7)
ax.fill_between(t_jan, janela_exemplo, 0, where=(janela_exemplo > 0), color='green', alpha=0.1, label='Energia Positiva')
ax.fill_between(t_jan, janela_exemplo, 0, where=(janela_exemplo < 0), color='red', alpha=0.1, label='Energia Negativa')
ax.axhline(rms_val, color='blue', linestyle='--', linewidth=2, label=f'RMS Global = {rms_val:.3f}')
ax.axhline(-rms_val, color='blue', linestyle='--', linewidth=2)
ax.set_title('[Domínio TEMPO] RMSE e Distribuição de Energia', fontweight='bold')
ax.legend()

# Explicativo da PSD das Band Power (Freq)
f, psd = sp_signal.welch(janela_exemplo, fs=fs, nperseg=512)
ax = axes[0, 1]
ax.plot(f, psd, color='black', linewidth=1)
ax.fill_between(f, psd, 0, where=(f>=0.5)&(f<=5), color='#3498db', alpha=0.4, label='LF (0.5-5Hz) [Ondas P/T]')
ax.fill_between(f, psd, 0, where=(f>=5)&(f<=15), color='#e74c3c', alpha=0.4, label='MF (5-15Hz) [Miolo QRS]')
ax.fill_between(f, psd, 0, where=(f>=15)&(f<=40), color='#2ecc71', alpha=0.4, label='HF (15-40Hz) [Arestas QRS]')
ax.set_xlim([0, 50])
ax.set_title('[Domínio FREQUÊNCIA] Band Power Distribution', fontweight='bold')
ax.legend()

# Explicativo da Wavelet (Tempo-Freq)
coeffs = pywt.wavedec(janela_exemplo, 'db4', level=4)
ax = axes[1, 0]
ax.plot(coeffs[0], color='purple', label='Aproximação cA4 (Visão Global Baixa Freq)')
ax.plot(coeffs[1] + 1.5, color='orange', label='Detalhes cD4 (Deslocado)')
ax.plot(coeffs[2] + 2.5, color='teal', label='Detalhes cD3 (Deslocado)')
ax.set_title('[Domínio TEMPO-FREQ] Coeficientes DWT (db4)', fontweight='bold')
ax.legend()
ax.set_xticks([]) # Omissão temporal pois sofrem downsampling diádico

# Explicativo Poincaré Lag-1 (Não linear)
ax = axes[1, 1]
ax.scatter(janela_exemplo[:-1], janela_exemplo[1:], s=5, alpha=0.5, color='crimson')
ax.plot([-1.5, 2.5], [-1.5, 2.5], 'k--', alpha=0.3, label='Linha de Identidade (x=y)')
ax.set_title('[Domínio NÃO-LINEAR] Pseudo-Poincaré (Lag 1 na série)', fontweight='bold')
ax.set_xlabel('Ponto $x_n$')
ax.set_ylabel('Ponto $x_{n+1}$')
ax.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'explicativo_grafico_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 6. Salvar Dataset Final Resultante

In [ ]:
output_csv = FIG_DIR.parent / 'dataset_features_extraidas.csv'
df_features.to_csv(output_csv, index=False)

print("=================================================================")
print(f"Dataset Ouro de Features gerado com sucesso!")
print(f"Caminho: {output_csv}")
print(f"Instâncias: {df_features.shape[0]} amostras de dados na linha (X)")
print(f"Colunas (Features Puras): {df_features.shape[1] - 3} features (y)")
print("=================================================================")

display(df_features.head(5).round(4))